# Multi index training
data intro:
- a total of 9 important indexes from the CN market including: 000016.SH, 000300.SH, 000852.SH, 000905.SH, 000985.CSI, 399303.SZ, 868008.WI, 8841425.WI, 932000.CSI

# import data

In [ ]:
import mysql.connector
import numpy as np
import pandas as pd
from pathlib import Path
import os

import qlib
from qlib.constant import REG_CN
from qlib.contrib.data.handler import Alpha158

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt


import lightgbm as lgb

from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

'''
import os
from dotenv import load_dotenv
'''

In [ ]:
# Connect to your MySQL database
import os
from dotenv import load_dotenv

load_dotenv()  # reads the .env file into environment variables


# Load a table into a DataFrame
if Path("bench_basic_data.parquet").exists():
    df = pd.read_parquet("bench_basic_data.parquet")
else:
    conn = mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        database=os.getenv("DB_NAME"),
    )
    df = pd.read_sql("SELECT * FROM bench_basic_data", conn)
    df.to_parquet("bench_basic_data.parquet", index=False)
    conn.close()
df.head()

# data inspection

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df['date'] = pd.to_datetime(df['date'])

In [ ]:
df.isnull().sum()
######## we found 4 null in dates!

In [ ]:
df[df['date'].isnull()]
######## i thought perhaps i can find out what these dates are so i did not drop them here
######## as it turns out i should just drop them in the first place

In [ ]:
df.groupby('code')['date'].agg(['min', 'max', 'count'])
####### okay we might need to delete 000905.SH

In [ ]:
df.loc[16550:16554]
####### i thought we can get out some info from the NAs
####### but it just shows how untidy this data set is

In [ ]:
df = df.sort_values(['code', 'date'])

In [ ]:
df[df.duplicated(subset=['code', 'date'], keep=False)]
df_clean = df.drop_duplicates(keep='first')
####### drop all duplicated rows
print(df.duplicated(keep=False).sum())
print(df_clean.duplicated(subset=['code', 'date'], keep=False).sum())
####### theres no duplicated dates left after we moved all duplicated rows


In [ ]:
print(df_clean.shape)
print(df_clean.head())

In [ ]:
df_clean.groupby('code')['date'].agg(['min', 'max', 'count'])

In [ ]:
# 可以试试这种写法, 比较高效发现异常
df_clean.query('code == "000905.SH" and date < "2018-01-02"')

In [ ]:
one_stock = df_clean[df_clean['code'] == '000905.SH'].sort_values('date')
print(one_stock['date'].head(10))
###### now we see where that weird date came about

In [ ]:
df_clean = df_clean.drop(df_clean[df_clean['date'] == '2000-09-05'].index)
len(df_clean)

In [ ]:
df_clean.groupby('code')['date'].agg(['min', 'max', 'count'])

In [ ]:
df_clean = df_clean.sort_values(['code', 'date'])

In [ ]:
print(df_clean.head(5))   
df_clean.groupby('code')['date'].apply(lambda x: x.is_monotonic_increasing)
# we checked the sorting but something went wrong

In [ ]:
# then we found that we forgot the NAs
######## this for loop can be turned into a function
for code in df_clean['code'].unique():
    sub = df_clean[df_clean['code'] == code].sort_values('date')
    if not sub['date'].is_monotonic_increasing:
        print(f"\n{code} has an issue:")
        diffs = sub['date'].diff()
        problem_rows = sub[diffs <= pd.Timedelta(0)]
        print(problem_rows)

In [ ]:
df_clean = df_clean.drop(df_clean[df_clean['date'].isnull()].index)
df_clean.groupby('code')['date'].agg(['min', 'max', 'count'])
# check again

In [ ]:
print(df_clean.groupby('code')['date'].apply(lambda x: x.is_monotonic_increasing)) 

# insert Alpha158
Qlib wants one stock per file

In [ ]:
output_dir = "raw_data"
os.makedirs(output_dir, exist_ok=True)

for code, group in df_clean.groupby('code'):
    out = group[['date', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME']].copy()
    out.columns = ['date', 'open', 'high', 'low', 'close', 'volume']
    out['factor'] = 1.0   # if you don't have a real adjustment factor field
    out['vwap'] = group['AMT'] / group['VOLUME']

    out = out.sort_values('date')
    out_path = os.path.join(output_dir, f"{code}.csv")
    out.to_csv(out_path, index=False)
    print(f"Saved {code}: {len(out)} rows")

im planning to do a walk-through or rolling valid
but since we are going to fit 3-4 models 
let's not use that until NN
right now this is just a baseline: Ridge and LightGBM

In [ ]:
qlib.init(provider_uri="qlib_data/", region=REG_CN)
########## so we still need to clear this before updating to github ###########

handler = Alpha158(
    start_time="2018-01-02",
    end_time="2026-06-05",
    fit_start_time="2018-01-02",
    fit_end_time="2023-12-29",
    instruments="all",
)

data = handler.fetch()
print(data.shape)

# model fitting

In [ ]:
# check NAs in the columns (we input VWAP so its not NA)
data.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
X =  data.iloc[:, 0:158]
y = data.iloc[:, 158]

In [ ]:
# drop all rows with NaN
keep = X.notna().all(axis=1) & y.notna()
X, y = X[keep], y[keep]
print(len(y))
print((X.isna().sum() == 0).sum(), y.isna().sum()) # check if we dropped every NA

In [ ]:
train_dates = slice("2018-01-02", "2023-12-29")
valid_dates = slice("2024-01-02", "2025-03-14")
test_dates  = slice("2025-03-17", "2026-06-05")

X_train, y_train = X.loc[train_dates], y.loc[train_dates]
X_valid, y_valid = X.loc[valid_dates], y.loc[valid_dates]
X_test,  y_test  = X.loc[test_dates],  y.loc[test_dates]

print(len(y_train) + len(y_valid) + len(y_test)) # check slicing

# Ridge

In [ ]:
best_alpha, best_score = None, np.inf
for alpha in np.logspace(-3, 3, 20):
    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train.values.ravel())
    preds = model.predict(X_valid)
    score = mean_squared_error(y_valid, preds)
    if score < best_score:
        best_alpha, best_score = alpha, score

final_model = Ridge(alpha=best_alpha)
final_model.fit(X_train, y_train.values.ravel())

In [ ]:
y.describe()

In [ ]:
test_preds = final_model.predict(X_test)

results = pd.DataFrame({"pred": test_preds, "actual": y_test.values}, index=y_test.index)

mse = mean_squared_error(results["actual"], results["pred"])
mae = (results["actual"] - results["pred"]).abs().mean()
pearson = results["pred"].corr(results["actual"])

daily_ic = results.groupby(level="datetime").apply(
    lambda g: g["pred"].corr(g["actual"], method="spearman")
)
rank_ic = daily_ic.mean()

print(f"Test MSE: {mse:.6f}")
print(f"Test MAE: {mae:.6f}")
print(f"Pearson: {pearson:.4f}")
print(f"Rank IC: {rank_ic:.4f}")

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test.index.get_level_values('datetime'), y_test.values, label="Actual", alpha=0.7)
plt.plot(y_test.index.get_level_values('datetime'), test_preds, label="Predicted", alpha=0.7)
plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# weird model behavior
let's take a closer look why

In [ ]:
window_mask = (X_test.index.get_level_values("datetime") >= "2025-08-01") & \
              (X_test.index.get_level_values("datetime") <= "2025-10-31")

X_window = X_test[window_mask]

# Find the max absolute value per column within this window
extreme_features = X_window.abs().max().sort_values(ascending=False)
print(extreme_features.head(20))

In [ ]:
print(X_window.loc[X_window[extreme_features.index[0]].abs() > 10])

from the points we suspect the original volume data has extreme outliers

In [ ]:
df_clean[df_clean['VOLUME'] < 100000]
######## compared to the summary of the VOLUME column below, we can tell the volumes these days
######## happen to be lower outliers, which indecates extreme values that are likely caused by
######## some special events

In [ ]:
df_clean.describe()

we plot the graph to get a better sense of what is happening

In [ ]:
for code in df_clean['code'].unique():
    sub = df_clean[df_clean['code'] == code].sort_values('date')
    
    plt.figure(figsize=(14, 4))
    plt.plot(sub['date'], sub['VOLUME'], linewidth=1)
    plt.axhline(sub['VOLUME'].quantile(0.25), color='gray', linestyle='--', linewidth=0.8, label='25th percentile')
    plt.title(f"{code} — Volume over time")
    plt.xlabel("Date")
    plt.ylabel("Volume")
    plt.legend()
    plt.tight_layout()
    plt.show()

from other data and events around the date, we assume that this is a data cleaning issue, probably unit change

In [ ]:
'''
932000.CSI
399303.SZ # changed ever since
000985.CSI
000300.SH
'''

for code in ['932000.CSI', '399303.SZ', '000985.CSI', '000300.SH']:
    sub = df_clean[df_clean['code'] == code].sort_values('date')

    print(sub[sub['VOLUME'] == min(sub['VOLUME'])])

In [ ]:
for code in ['932000.CSI', '399303.SZ', '000985.CSI', '000300.SH']:
    sub = df_clean[df_clean['code'] == code].sort_values('date')
    sub = sub[('2025-08-01' <= sub['date']) & (sub['date'] <= '2025-08-31')]
    
    plt.figure(figsize=(14, 4))
    plt.plot(sub['date'], sub['VOLUME'], marker='.', markersize=4, linewidth=1)
    plt.title(f"{code} — Volume over time")
    plt.xlabel("Date")
    plt.ylabel("Volume")
    plt.legend()
    plt.tight_layout()
    plt.show()

see the magnitue

In [ ]:
for code in ['932000.CSI', '399303.SZ', '000985.CSI', '000300.SH']:
    sub = df_clean[df_clean['code'] == code].sort_values('date')

    vol_aug01 = sub[sub['date'] == '2025-08-01']['VOLUME'].values[0]
    vol_aug04 = sub[sub['date'] == '2025-08-04']['VOLUME'].values[0]

    print(vol_aug01 / vol_aug04)
######## all around 1000000
######## now lets time it back

see if by multiplying 1_000_000 can do

In [ ]:
for code in ['932000.CSI', '000985.CSI', '000300.SH']:
    sub = df_clean[df_clean['code'] == code].sort_values('date')
    sub = sub[('2025-08-01' <= sub['date']) & (sub['date'] <= '2025-08-31')]

    fix_dates = ['2025-08-04', '2025-08-05', '2025-08-06','2025-08-07','2025-08-08','2025-08-11']
    sub.loc[sub['date'].isin(fix_dates), 'VOLUME'] *= 1000000

    plt.figure(figsize=(14, 4))
    plt.plot(sub['date'], sub['VOLUME'], marker='.', markersize=4, linewidth=1)
    plt.title(f"{code} — Volume over time")
    plt.xlabel("Date")
    plt.ylabel("Volume")
    plt.legend()
    plt.tight_layout()
    plt.show()
###### yep now we know how to fix these data points

# 399303.SZ
sub = df_clean[df_clean['code'] == '399303.SZ'].sort_values('date')
sub.loc[sub['date'] >= '2025-08-04', 'VOLUME'] *= 1000000

plt.figure(figsize=(14, 4))
plt.plot(sub['date'], sub['VOLUME'], linewidth=1)
plt.title("399303.SZ — Volume over time")
plt.xlabel("Date")
plt.ylabel("Volume")
plt.legend()
plt.tight_layout()
plt.show()

####### then i double checked, it is the UNIT problem
    

now we've made sure this will work (also comparing to other data sources)
we change the original data

In [ ]:
######## data cleaning
for code in ['932000.CSI', '000985.CSI', '000300.SH']:
    sub = df_clean[df_clean['code'] == code].sort_values('date')

    fix_dates = ['2025-08-04', '2025-08-05', '2025-08-06','2025-08-07','2025-08-08','2025-08-11']
    sub.loc[sub['date'].isin(fix_dates), 'VOLUME'] *= 1000000

    df_clean[df_clean['code'] == code] = sub

# 399303.SZ
sub = df_clean[df_clean['code'] == '399303.SZ'].sort_values('date')
sub.loc[sub['date'] >= '2025-08-04', 'VOLUME'] *= 1000000
df_clean[df_clean['code'] == '399303.SZ'] = sub


In [ ]:
# check
for code in df_clean['code'].unique():
    sub = df_clean[df_clean['code'] == code].sort_values('date')
    
    plt.figure(figsize=(14, 4))
    plt.plot(sub['date'], sub['VOLUME'], linewidth=1)
    plt.axhline(sub['VOLUME'].quantile(0.25), color='gray', linestyle='--', linewidth=0.8, label='25th percentile')
    plt.title(f"{code} — Volume over time")
    plt.xlabel("Date")
    plt.ylabel("Volume")
    plt.legend()
    plt.tight_layout()
    plt.show()

take a look at the peaks to see if its also a data flaw

In [ ]:
for code in df_clean['code'].unique():
    sub = df_clean[df_clean['code'] == code].sort_values('date')

    print(sub[sub['VOLUME'] == max(sub['VOLUME'])])

####### okay this will show when they all reached the highest volume
####### just to see if the peak is a unit problem too
####### 24/10/08 is a special day for the cn stock market

# y = 0 model

In [ ]:
####### this is to check if the model is better than predicting everything as 0
test_preds_mlp = zeros = np.zeros(len(X_test))
pred_series_mlp = pd.Series(test_preds_mlp, index=X_test.index)

# MSE
mse_0 = np.mean((pred_series_mlp - y_test) ** 2)
# MAE
mae_0 = np.mean(np.abs(pred_series_mlp - y_test))

print('mse:', mse_0, 'mae:', mae_0)

# alpha158 and ridge AGAIN

In [ ]:
# do the alpha and input ALL OVER AGAIN :((((
output_dir = "cleaned_raw_data"
os.makedirs(output_dir, exist_ok=True)

for code, group in df_clean.groupby('code'):
    out = group[['date', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME']].copy()
    out.columns = ['date', 'open', 'high', 'low', 'close', 'volume']
    out['factor'] = 1.0   # if you don't have a real adjustment factor field
    out['vwap'] = group['AMT'] / group['VOLUME']

    out = out.sort_values('date')
    out_path = os.path.join(output_dir, f"{code}.csv")
    out.to_csv(out_path, index=False)
    print(f"Saved {code}: {len(out)} rows")

In [ ]:
qlib.init(provider_uri="qlib_data/", region=REG_CN)

handler = Alpha158(
    start_time="2018-01-02",
    end_time="2026-06-05",
    fit_start_time="2018-01-02",
    fit_end_time="2023-12-29",
    instruments=['000016.SH', '000300.SH', '000852.SH', '000905.SH', '000985.CSI',
       '399303.SZ', '868008.WI', '8841425.WI', '932000.CSI'],
)

data = handler.fetch()
print(data.shape)
print(data.head())

In [ ]:
# data splitting
X =  data.iloc[:, 0:158]
y = data.iloc[:, 158]

keep = X.notna().all(axis=1) & y.notna()
X, y = X[keep], y[keep]

train_dates = slice("2018-01-02", "2023-12-29")
valid_dates = slice("2024-01-02", "2025-03-14")
test_dates  = slice("2025-03-17", "2026-06-05")

X_train, y_train = X.loc[train_dates], y.loc[train_dates]
X_valid, y_valid = X.loc[valid_dates], y.loc[valid_dates]
X_test,  y_test  = X.loc[test_dates],  y.loc[test_dates]

# Rige
best_alpha, best_score = None, np.inf
for alpha in np.logspace(-3, 3, 20):
    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train.values.ravel())
    preds = model.predict(X_valid)
    score = mean_squared_error(y_valid, preds)
    if score < best_score:
        best_alpha, best_score = alpha, score

print('best alpha:', best_alpha)

final_model = Ridge(alpha=best_alpha)
final_model.fit(X_train, y_train.values.ravel())

# evaluate
test_preds = final_model.predict(X_test)

results = pd.DataFrame({"pred": test_preds, "actual": y_test.values}, index=y_test.index)

mse = mean_squared_error(results["actual"], results["pred"])
mae = (results["actual"] - results["pred"]).abs().mean()
pearson = results["pred"].corr(results["actual"])

daily_ic = results.groupby(level="datetime").apply(
    lambda g: g["pred"].corr(g["actual"], method="spearman")
)
rank_ic = daily_ic.mean()

print('MSE:', mse, 'MAE:', mae, 'corr:', pearson, 'rank_ic:', rank_ic)

In [ ]:
plt.figure(figsize=(12, 5))
plt.scatter(y_test.index.get_level_values('datetime'), y_test.values, s = 3, label="Actual", alpha=0.7, linewidth=0.8)
plt.scatter(y_test.index.get_level_values('datetime'), test_preds, s = 3, label="Predicted", alpha=0.7, linewidth=0.8)
plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

'''
it might be getting the general up/down direction of returns right in raw-value terms slightly better than 
it's getting the relative ranking across stocks right on any given day
'''

# LightGBM

In [ ]:
# LightGBM
train_set = lgb.Dataset(X_train, label=y_train)
valid_set = lgb.Dataset(X_valid, label=y_valid, reference=train_set)

params = {
    "objective": "regression",
    "metric": "mse",
    "learning_rate": 0.01,
    "num_leaves": 31,
    "max_depth": -1,
    "verbose": -1,
}

lgb_model = lgb.train(
    params,
    train_set,
    num_boost_round=500,
    valid_sets=[valid_set],
    callbacks=[lgb.early_stopping(stopping_rounds=20)] # watches validation loss
)

# get the important parameters 
importances = pd.Series(lgb_model.feature_importance(importance_type="gain"),
                         index=X_train.columns)
print(importances.sort_values(ascending=False).head(10))

test_preds_lgb = lgb_model.predict(X_test, num_iteration = lgb_model.best_iteration)
pred_series_lgb = pd.Series(test_preds_lgb, index=X_test.index)

results_lgb = pd.DataFrame({"pred": test_preds_lgb, "actual": y_test.values}, index=y_test.index)

# MSE
mse_lgb = np.mean((pred_series_lgb - y_test) ** 2)
# MAE
mae_lgb = np.mean(np.abs(pred_series_lgb - y_test))
# pearson corr
p_corr_lgb = pred_series_lgb.corr(y_test)
# rank ic
daily_ic = results_lgb.groupby(level="datetime").apply(
    lambda g: g["pred"].corr(g["actual"], method="spearman")
)
rank_ic = daily_ic.mean()

print('MSE:', mse_lgb, 'MAE:', mae_lgb, 'corr:', p_corr_lgb, 'rank_ic:', rank_ic)

In [ ]:
plt.figure(figsize=(12, 5))
plt.scatter(y_test.index.get_level_values('datetime'), y_test.values, s = 3, label="Actual", alpha=0.7, linewidth=0.8)
plt.scatter(y_test.index.get_level_values('datetime'), test_preds_lgb, s = 3, label="Predicted", alpha=0.7, linewidth=0.8)
plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# MLP
- grid and optimizer for tuning

In [ ]:
# MLP
param_grid = [
    {"hidden_layer_sizes": (64, 32, 16), "alpha": 0.01},
    {"hidden_layer_sizes": (128, 64), "alpha": 0.01},
    {"hidden_layer_sizes": (128, 64), "alpha": 0.001},
    {"hidden_layer_sizes": (128, 64, 32), "alpha": 0.001},
]

for para in param_grid:
    mlp_model = MLPRegressor(
        hidden_layer_sizes= para["hidden_layer_sizes"],
        activation="relu",
        solver="adam",
        alpha= para['alpha'],                    
        learning_rate_init=0.001,
        max_iter=1000,
        early_stopping=False,     # early stopping will cause data leakage!!!      
        random_state=42,  
    )

    mlp_model.fit(X_train, y_train)

    test_preds_mlp = mlp_model.predict(X_valid)
    pred_series_mlp = pd.Series(test_preds_mlp, index=X_valid.index)

    results_mlp = pd.DataFrame({"pred": test_preds_mlp, "actual": y_valid.values}, index=y_valid.index)

    # MSE
    mse_mlp = np.mean((pred_series_mlp - y_valid) ** 2)
    # MAE
    mae_mlp = np.mean(np.abs(pred_series_mlp - y_valid))
    # pearson corr
    p_corr_mlp = pred_series_mlp.corr(y_valid)
    # rank ic
    daily_ic_mlp = results_mlp.groupby(level="datetime").apply(
        lambda g: g["pred"].corr(g["actual"], method="spearman")
    )
    rank_ic_mlp = daily_ic_mlp.mean()

    print('MSE:', mse_mlp, 'MAE:', mae_mlp, 'corr:', p_corr_mlp, 'rank_ic:', rank_ic_mlp)
# {"hidden_layer_sizes": (128, 64), "alpha": 0.01} produced the best result

stability check: set seed to different numbers then check if the model still performs well

In [ ]:
ic = []
for seed in [42, 5, 47, 467, 76, 15]:
    mlp_model = MLPRegressor(
        hidden_layer_sizes= (64, 32, 16),
        activation="relu",
        solver="adam",
        alpha= 0.01,
        random_state=seed,                    
        max_iter=1000,
        early_stopping=False,        
    )
    mlp_model.fit(X_train, y_train)

    test_preds_mlp = mlp_model.predict(X_valid)                    
    pred_series_mlp = pd.Series(test_preds_mlp, index=X_valid.index)

    results_mlp = pd.DataFrame({"pred": test_preds_mlp, "actual": y_valid.values}, index=y_valid.index)

    # MSE
    mse_mlp = np.mean((pred_series_mlp - y_valid) ** 2)
    # MAE
    mae_mlp = np.mean(np.abs(pred_series_mlp - y_valid))
    # pearson corr
    p_corr_mlp = pred_series_mlp.corr(y_valid)
    # rank ic
    daily_ic_mlp = results_mlp.groupby(level="datetime").apply(
        lambda g: g["pred"].corr(g["actual"], method="spearman")
    )
    rank_ic_mlp = daily_ic_mlp.mean()
    ic.append(rank_ic_mlp)

    print(seed, ':', 'MSE:', mse_mlp, 'MAE:', mae_mlp, 'corr:', p_corr_mlp, 'rank_ic:', rank_ic_mlp)

print('mean:', np.mean(ic), 'std:', np.std(ic))

use Bayesian Optimization to do parameter tunning

In [ ]:
import optuna

In [ ]:
def objective(trial):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    hidden_sizes = tuple(
        trial.suggest_int(f"units_l{i}", 16, 128, step=16) for i in range(n_layers)
    )
    alpha = trial.suggest_float("alpha", 1e-4, 0.5, log=True)   # log scale, same reasoning as your Ridge alpha grid
    learning_rate_init = trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True)
    
    model = MLPRegressor(
        hidden_layer_sizes=hidden_sizes,
        alpha=alpha,
        learning_rate_init=learning_rate_init,
        activation="relu",
        solver="adam",
        max_iter=1000,
        early_stopping=False,
        random_state=42,
    )
    # ... same param sampling ...
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)                   
    pred_series_mlp = pd.Series(preds, index=X_valid.index)

    results_mlp = pd.DataFrame({"pred": pred_series_mlp, "actual": y_valid.values}, index=y_valid.index)

    daily_ic_mlp = results_mlp.groupby(level="datetime").apply(
        lambda g: g["pred"].corr(g["actual"], method="spearman")
    )
    rank_ic_mlp = daily_ic_mlp.mean()

    return rank_ic_mlp

study = optuna.create_study(direction="maximize")   # maximize Rank IC

study.optimize(objective, n_trials=50)   # 50 intelligent trials instead of a fixed grid

print("Best params:", study.best_params)
print("Best valid rank_ic:", study.best_value)

In [ ]:
ic = []
for seed in [41, 3, 67, 248, 65, 86]:
    mlp_model = MLPRegressor(
        hidden_layer_sizes= (80),
        activation="relu",
        solver="adam",
        alpha= 0.1511297638395075,
        random_state=seed,                    
        learning_rate_init= 0.00030892943523035875,
        max_iter=1000,
        early_stopping=False,        
    )
    mlp_model.fit(X_train, y_train)

    test_preds_mlp = mlp_model.predict(X_valid)                     ###### used valid to check, test set actually performs even worse
    pred_series_mlp = pd.Series(test_preds_mlp, index=X_valid.index)

    results_mlp = pd.DataFrame({"pred": test_preds_mlp, "actual": y_valid.values}, index=y_valid.index)

    # MSE
    mse_mlp = np.mean((pred_series_mlp - y_valid) ** 2)
    # MAE
    mae_mlp = np.mean(np.abs(pred_series_mlp - y_valid))
    # pearson corr
    p_corr_mlp = pred_series_mlp.corr(y_valid)
    # rank ic
    daily_ic_mlp = results_mlp.groupby(level="datetime").apply(
        lambda g: g["pred"].corr(g["actual"], method="spearman")
    )
    rank_ic_mlp = daily_ic_mlp.mean()
    ic.append(rank_ic_mlp)

    print(seed, ':', 'MSE:', mse_mlp, 'MAE:', mae_mlp, 'corr:', p_corr_mlp, 'rank_ic:', rank_ic_mlp)
print('mean_ic:', np.mean(ic), 'std_ic', np.std(ic))

compare the 2 tuning result
- grid tunning: mean: -0.00856873822975518 std: 0.026337495376461406  # this has no valid set (but likely not the reson why it performs so badly because the model itself doesn't)
- model tunning: mean_ic: 0.011883239171374765 std_ic 0.019036297248195225 # but to mention this is after running the valid set

In [ ]:
# final result on test set:
# and we are going to train the model on both train and valid set combined

mlp_model = MLPRegressor(
    hidden_layer_sizes= (80),
    activation="relu",
    solver="adam",
    alpha= 0.1511297638395075,
    random_state=42,  
    learning_rate_init= 0.00030892943523035875,                  
    max_iter=1000,
    early_stopping=False,        
)

X_train_valid = pd.concat([X_train, X_valid], axis=0)
y_train_valid = pd.concat([y_train, y_valid], axis=0)

X_train_valid = X_train_valid.sort_index(level='datetime')
y_train_valid = y_train_valid.sort_index(level='datetime')

mlp_model.fit(X_train_valid, y_train_valid)

test_preds_mlp = mlp_model.predict(X_test)                    
pred_series_mlp = pd.Series(test_preds_mlp, index=X_test.index)

results_mlp = pd.DataFrame({"pred": test_preds_mlp, "actual": y_test.values}, index=y_test.index)

# MSE
mse_mlp = np.mean((pred_series_mlp - y_test) ** 2)
# MAE
mae_mlp = np.mean(np.abs(pred_series_mlp - y_test))
# pearson corr
p_corr_mlp = pred_series_mlp.corr(y_test)
# rank ic
daily_ic_mlp = results_mlp.groupby(level="datetime").apply(
    lambda g: g["pred"].corr(g["actual"], method="spearman")
)
rank_ic_mlp = daily_ic_mlp.mean()
ic.append(rank_ic_mlp)

print(seed, ':', 'MSE:', mse_mlp, 'MAE:', mae_mlp, 'corr:', p_corr_mlp, 'rank_ic:', rank_ic_mlp)


In [ ]:
plt.figure(figsize=(12, 5))
plt.scatter(y_test.index.get_level_values('datetime'), y_test.values, s = 3, label="Actual", alpha=0.7, linewidth=0.8)
plt.scatter(y_test.index.get_level_values('datetime'), test_preds_mlp, s = 3, label="Predicted", alpha=0.7, linewidth=0.8)
plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

we can officially conclude that MLP is not suitable for this data set AT ALL
lets just try LSTM for project completeness

# LSTM

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
# build simple features
raw_df = df_clean.sort_index()
raw_df["log_ret"] = raw_df.groupby('code')['CLOSE'].transform(lambda x: np.log(x / x.shift(1)))
raw_df["vol_z"] = raw_df.groupby('code')["VOLUME"].transform(lambda x: (x-x.rolling(20).mean()) / x.rolling(20).std())
raw_df["hl_range"] = (raw_df["HIGH"] - raw_df["LOW"]) / raw_df["CLOSE"]
raw_df['STD20'] = raw_df.groupby('code')['CLOSE'].transform(lambda x: x.rolling(20).std() / x)

def resi30(close_series, window=30):
    resid = pd.Series(index=close_series.index, dtype=float)
    values = close_series.values
    for i in range(window - 1, len(values)):
        y = values[i - window + 1: i + 1]
        x = np.arange(window)
        # fit linear trend: y = a*x + b
        a, b = np.polyfit(x, y, 1)
        fitted_today = a * (window - 1) + b
        resid.iloc[i] = (values[i] - fitted_today) / values[i]
    return resid

raw_df['RESI30'] = raw_df.groupby('code')['CLOSE'].transform(lambda x: resi30(x))

def imin20(close_series, window=20):
    result = pd.Series(index=close_series.index, dtype=float)
    values = close_series.values
    for i in range(window - 1, len(values)):
        window_vals = values[i - window + 1: i + 1]
        min_pos_from_start = np.argmin(window_vals)          # 0 = start of window, window-1 = today
        days_ago = (window - 1) - min_pos_from_start          # 0 = today, window-1 = window start
        result.iloc[i] = days_ago / window
    return result

raw_df['IMIN20'] = raw_df.groupby('code')['CLOSE'].transform(lambda x: imin20(x))



# Label: next-day-after-next close-to-close return (realizable, per earlier discussion)
raw_df["label"] = raw_df.groupby('code')['CLOSE'].transform(lambda x: x.shift(-2) / x.shift(-1) - 1)

raw_df = raw_df.dropna()
raw_df = raw_df.drop(['PCT_CHG', 'AMT'], axis = 1)

feature_cols = ["log_ret", "vol_z", "hl_range", "STD20", "RESI30", "IMIN20"]

raw_df.head()

In [ ]:
raw_df.describe()

In [ ]:
print(raw_df["label"].describe()) 
####### this is the raw return, different from Alpha158

In [ ]:
# build sliding window
window = 20

X_seq, y_seq, dates_seq, codes_seq = [], [], [], []

for code, group in raw_df.groupby('code'):
    group = group.sort_values('date')
    values = group[feature_cols].values
    labels = group['label'].values
    dates = group['date'].values
    
    for i in range(window, len(group)):
        X_seq.append(values[i-window:i])
        y_seq.append(labels[i])
        dates_seq.append(dates[i])
        codes_seq.append(code)

X_seq = np.array(X_seq)   
y_seq = np.array(y_seq)
dates_seq = np.array(dates_seq)
codes_seq = np.array(codes_seq)

In [ ]:
# test/valid/train split
dates_seq_pd = pd.to_datetime(dates_seq)   # ensure comparable datetime type

train_mask = (dates_seq_pd >= "2018-01-02") & (dates_seq_pd <= "2023-12-29")
valid_mask = (dates_seq_pd >= "2024-01-02") & (dates_seq_pd <= "2025-03-14")
test_mask  = (dates_seq_pd >= "2025-03-17") & (dates_seq_pd <= "2026-06-05")

X_train, y_train = X_seq[train_mask], y_seq[train_mask]
X_valid, y_valid = X_seq[valid_mask], y_seq[valid_mask]
X_test, y_test = X_seq[test_mask], y_seq[test_mask]
dates_test = dates_seq[test_mask]
codes_test = codes_seq[test_mask]

In [ ]:
# 5. Simple LSTM model
class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        last_hidden = h_n[-1]          # final layer's hidden state
        return self.fc(last_hidden).squeeze(-1)

model = LSTMRegressor(input_size=len(feature_cols))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # weight_decay = L2 reg
criterion = nn.MSELoss()

In [ ]:
# 6. Training loop with manual early stopping on valid loss

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_valid_t = torch.tensor(X_valid, dtype=torch.float32)
y_valid_t = torch.tensor(y_valid, dtype=torch.float32)

best_valid_loss = float("inf")
patience, patience_counter = 20, 0
best_state = None

for epoch in range(500):
    model.train()
    optimizer.zero_grad()
    preds = model(X_train_t)
    loss = criterion(preds, y_train_t)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        valid_loss = criterion(model(X_valid_t), y_valid_t).item()

# okay lets check
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, valid_loss={valid_loss:.5f}")

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state)

In [ ]:
# 7. Evaluate on test set
model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    test_preds = model(X_test_tensor).numpy()

test_index = pd.MultiIndex.from_arrays([dates_test, codes_test], names=["datetime", "instrument"])

eval_df = pd.DataFrame({"pred": test_preds, "actual": y_test}, index=test_index)

daily_ic = eval_df.groupby(level="datetime").apply(
    lambda g: g["pred"].corr(g["actual"], method="spearman")
)
rank_ic = daily_ic.mean()

mse = np.mean((y_test - test_preds) ** 2)
mae = np.mean(np.abs(y_test - test_preds))
corr = pd.Series(test_preds).corr(pd.Series(y_test))

print(f"Test MSE: {mse:.6f}")
print(f"Test MAE: {mae:.6f}")
print(f"Pearson corr: {corr:.4f}")
print(f"Rank IC: {rank_ic:.4f}")

In [ ]:
plt.figure(figsize=(12, 5))
plt.scatter(test_index.get_level_values('datetime'), y_test, s = 3, label="Actual", alpha=0.7)
plt.scatter(test_index.get_level_values('datetime'), test_preds, s = 3, label="Predicted", alpha=0.7)
plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
plt.xlabel("Date")
plt.ylabel("Return")
plt.ylim(-0.05, 0.05)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# lest check if the model recognizes a genral pattern
seeds = [0, 1, 42, 123, 7]
rank_ics = []

for seed in seeds:
    torch.manual_seed(seed)   # PyTorch's equivalent of sklearn's random_state
    
    model = LSTMRegressor(input_size=len(feature_cols))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    criterion = nn.MSELoss()
    
    best_valid_loss = float("inf")
    patience, patience_counter = 20, 0
    best_state = None

    for epoch in range(400):
        model.train()
        optimizer.zero_grad()
        preds = model(X_train_t)
        loss = criterion(preds, y_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            valid_loss = criterion(model(X_valid_t), y_valid_t).item()

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    
    model.eval()
    with torch.no_grad():
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
        test_preds = model(X_test_tensor).numpy()

    test_index = pd.MultiIndex.from_arrays([dates_test, codes_test], names=["datetime", "instrument"])
    eval_df = pd.DataFrame({"pred": test_preds, "actual": y_test}, index=test_index)
    
    ic = eval_df.groupby(level="datetime").apply(lambda g: g["pred"].corr(g["actual"], method="spearman")).mean()

    rank_ics.append(ic)
    print(seed, ic)

print(f"\nMean: {np.mean(rank_ics):.4f}, Std: {np.std(rank_ics):.4f}")

- y = 0:
    - mse: 0.00018692912232227743 mae: 0.00962342807353104
- Ridge: 
    - MSE: 0.00018983546760864556 MAE: 0.009768844 corr: 0.06227176655449924 rank_ic: 0.016271186440677963
- LightGBM: 
    - MSE: 0.00018698142311200706 MAE: 0.0096016180204029 corr: 0.008931415230085752 rank_ic: 0.0003920664536337951
- MLP:
    - MSE: 0.00020431798 MAE: 0.01011178 corr: 0.027972189142707073 rank_ic: 0.0080225988700565
    - rank_ic stability: mean_ic: 0.011883239171374765 std_ic 0.019036297248195225
- LSTM: 
    - Test MSE: 0.000238, Test MAE: 0.009832, Pearson corr: 0.0010, Rank IC: 0.0259 (one round test)
    - rank_ic stability: Mean: 0.0128, Std: 0.0199

by comparison, Ridge is probably the best model
- next steps may include:
    - walk-through validation
    - protifolio construction then validate model usefullness